In [1]:
import pandas as pd 
import numpy as np
import joblib
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from io import StringIO

In [3]:
dfm=pd.read_csv("cleaned_integrated_materials.csv")

In [4]:
dfm.head()

,Material ID,Material Type,Supplier Region,Suitable Product Categories,Recommended Packaging Use Cases,Packaging Type,Recyclability Category,Recyclability (%),Recycled Content (%),Reusability (%),...,Carbon Footprint (kg CO2/unit),CO2 Emission per kg (estimated),Cost per Unit (USD),Waste Reduction Impact (%),Load Handling Score,Moisture Resistance Score,Thermal Resistance Score,Supplier Sustainability Compliance (%),Annual Usage (units),Total Material Weight (tons)
0,MAT_0001,Cardboard,EMEA,"E-commerce, Food & Beverage, Consumer Goods, A...",Last-mile delivery and primary e-commerce pack...,Cardboard Boxes,High,98.0,79.0,49.0,...,0.78,0.54,2.24,61.0,6.0,5.0,4.0,85.0,101260.0,790.0
1,MAT_0002,Paper/Bio-Based,APAC,"Fragile Items, Cosmetics, Pharmaceuticals, Int...",Void-fill and cushioning for fragile products,Protective Fillers (Paper/Biodegradable),High,100.0,93.0,31.0,...,0.52,0.32,1.82,68.0,3.0,3.0,3.0,88.0,73961.0,545.0
2,MAT_0003,Steel,AMERICAS,"Heavy Industrial Components, High-Security Goods","Secure, high-load international shipping and l...",Steel Racks & Containers,High,87.0,78.0,100.0,...,3.67,3.25,25.00,85.0,8.0,9.0,9.0,88.0,14649.0,4979.0
3,MAT_0004,Paper/Bio-Based,ROW,"Fragile Items, Cosmetics, Pharmaceuticals, Int...",Void-fill and cushioning for fragile products,Protective Fillers (Paper/Biodegradable),High,100.0,89.0,31.0,...,0.48,0.31,2.43,69.0,1.0,1.0,4.0,81.0,75036.0,656.0
4,MAT_0005,Paper/Bio-Based,LATAM,"Fragile Items, Cosmetics, Pharmaceuticals, Int...",Void-fill and cushioning for fragile products,Protective Fillers (Paper/Biodegradable),High,100.0,92.0,28.0,...,0.41,0.31,1.69,61.0,3.0,2.0,3.0,81.0,69209.0,549.0


In [7]:
ohe_cols=['Material Type', 'Supplier Region', 'Suitable Product Categories', 'Recommended Packaging Use Cases', 'Packaging Type', 'Recyclability Category']
ohe_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_data = ohe_encoder.fit_transform(dfm[ohe_cols])

In [8]:
ohe_feature_names = ohe_encoder.get_feature_names_out(ohe_cols)
df_ohe = pd.DataFrame(ohe_data, columns=ohe_feature_names)

In [10]:
df_encoded = pd.concat([dfm.drop(columns=ohe_cols), df_ohe], axis=1)

In [11]:
df_encoded.head()

,Material ID,Recyclability (%),Recycled Content (%),Reusability (%),Biodegradation Time (days),Carbon Footprint (kg CO2/unit),CO2 Emission per kg (estimated),Cost per Unit (USD),Waste Reduction Impact (%),Load Handling Score,...,Recommended Packaging Use Cases_Surface protection during transit; minimal void-fill,Recommended Packaging Use Cases_Void-fill and cushioning for fragile products,Packaging Type_Bubble Wrap (Minimal Use),Packaging Type_Cardboard Boxes,Packaging Type_Foldable & Stackable Containers,Packaging Type_Plastic Totes & Pallets,Packaging Type_Protective Fillers (Paper/Biodegradable),Packaging Type_Steel Racks & Containers,Recyclability Category_High,Recyclability Category_Medium
0,MAT_0001,98.0,79.0,49.0,188.0,0.78,0.54,2.24,61.0,6.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,MAT_0002,100.0,93.0,31.0,65.0,0.52,0.32,1.82,68.0,3.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,MAT_0003,87.0,78.0,100.0,180208.0,3.67,3.25,25.00,85.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,MAT_0004,100.0,89.0,31.0,65.0,0.48,0.31,2.43,69.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,MAT_0005,100.0,92.0,28.0,122.0,0.41,0.31,1.69,61.0,3.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


# Normalize Numeric Features

In [12]:
scaling_cols=  ['Recyclability (%)', 'Recycled Content (%)', 'Reusability (%)', 'Biodegradation Time (days)', 'Carbon Footprint (kg CO2/unit)', 'CO2 Emission per kg (estimated)', 'Cost per Unit (USD)', 'Waste Reduction Impact (%)', 'Load Handling Score', 'Moisture Resistance Score', 'Thermal Resistance Score', 'Supplier Sustainability Compliance (%)', 'Annual Usage (units)', 'Total Material Weight (tons)']

In [13]:
scaler = MinMaxScaler()
df_encoded[scaling_cols] = scaler.fit_transform(df_encoded[scaling_cols])

In [14]:
joblib.dump(ohe_encoder, 'ohe_encoder.pkl')
joblib.dump(scaler, 'numeric_scaler.pkl')
print(f"Saved One-Hot Encoder to {'ohe_encoder.pkl'}")
print(f"Saved MinMaxScaler to {'numeric_scaler.pkl'}")

Saved One-Hot Encoder to ohe_encoder.pkl
Saved MinMaxScaler to numeric_scaler.pkl


In [15]:
encoding_report = f"""
# Encoding and Normalization Report
---

## Encoding Strategy
1. **Nominal Categories** ({', '.join(ohe_cols)}): Used **One-Hot Encoding** (OHE) to create binary features for each unique category, as per the project plan. The fitted encoder object has been saved for reuse during model inference.

## Normalization Strategy
1. **Numeric Features** ({', '.join(scaling_cols)}): Used **MinMaxScaler** to scale all features to the range [0, 1]. This is suitable for features with different unit scales (e.g., percentages, scores, cost). The fitted scaler object has been saved for reuse during model inference.

## Validation (Final Encoded Dataset)
* **No missing values** remaining.
* **Numeric features** scaled between 0 and 1.
* **Categorical fields** successfully converted to numeric binary features.
"""

# Save Encoding/Normalization Report (Deliverable)
with open('encoding_normalization_report.md', 'w') as f:
    f.write(encoding_report)
print(f"Generated encoding/normalization report: {'encoding_normalization_report.md'}")

Generated encoding/normalization report: encoding_normalization_report.md


In [16]:
df_encoded.head()

,Material ID,Recyclability (%),Recycled Content (%),Reusability (%),Biodegradation Time (days),Carbon Footprint (kg CO2/unit),CO2 Emission per kg (estimated),Cost per Unit (USD),Waste Reduction Impact (%),Load Handling Score,...,Recommended Packaging Use Cases_Surface protection during transit; minimal void-fill,Recommended Packaging Use Cases_Void-fill and cushioning for fragile products,Packaging Type_Bubble Wrap (Minimal Use),Packaging Type_Cardboard Boxes,Packaging Type_Foldable & Stackable Containers,Packaging Type_Plastic Totes & Pallets,Packaging Type_Protective Fillers (Paper/Biodegradable),Packaging Type_Steel Racks & Containers,Recyclability Category_High,Recyclability Category_Medium
0,MAT_0001,0.955556,0.794521,0.474227,0.000871,0.139881,0.087542,0.078866,0.569231,0.555556,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,MAT_0002,1.000000,0.986301,0.288660,0.000189,0.062500,0.013468,0.062549,0.676923,0.222222,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,MAT_0003,0.711111,0.780822,1.000000,1.000000,1.000000,1.000000,0.963092,0.938462,0.777778,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,MAT_0004,1.000000,0.931507,0.288660,0.000189,0.050595,0.010101,0.086247,0.692308,0.000000,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,MAT_0005,1.000000,0.972603,0.257732,0.000505,0.029762,0.010101,0.057498,0.569231,0.222222,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [17]:
df_encoded.to_csv('materials_final_encoded.csv',index=False)